In [0]:
df_bronze = spark.readStream.table(
    "workspace.lab_db.bronze_orders"
)

In [0]:
from pyspark.sql.functions import col

df_silver = df_bronze.filter(
    col("customer_id").isNotNull() &
    (col("amount") > 0)
)

In [0]:
from pyspark.sql.functions import col

df_silver = (
    df_bronze
    .filter(
        col("customer_id").isNotNull() &
        (col("amount") > 0)
    )
    .withWatermark("order_timestamp", "10 minutes")
    .dropDuplicates(["order_id", "order_timestamp"])
)

In [0]:
from pyspark.sql.functions import col, to_timestamp

df_silver = (
    df_bronze
    .withColumn(
        "order_timestamp",
        to_timestamp(col("order_timestamp"))
    )
    .filter(
        col("customer_id").isNotNull() &
        (col("amount") > 0)
    )
    .withWatermark("order_timestamp", "10 minutes")
    .dropDuplicates(["order_id", "order_timestamp"])
)

In [0]:
df_silver.printSchema()

In [0]:
from pyspark.sql.functions import col, to_timestamp

df_silver = (
    df_bronze
    .withColumn(
        "order_timestamp",
        to_timestamp(col("order_timestamp"))
    )
    .withColumn(
        "amount",
        col("amount").cast("double")
    )
    .filter(
        col("customer_id").isNotNull() &
        (col("amount") > 0)
    )
    .withWatermark("order_timestamp", "10 minutes")
    .dropDuplicates(["order_id", "order_timestamp"])
)

In [0]:
df_silver.printSchema()

In [0]:
(
    df_silver.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/lab_db/lab_landing/checkpoints/silver_orders_v2/"
    )
    .outputMode("append")
    .trigger(availableNow=True)
    .table("workspace.lab_db.silver_orders")
)

In [0]:
spark.sql("""
SELECT *
FROM workspace.lab_db.silver_orders
ORDER BY order_id
""").show(truncate=False)

In [0]:
spark.sql("""
DESCRIBE workspace.lab_db.silver_orders
""").show(truncate=False)